In [4]:
# imports and env
import base64
from dotenv import load_dotenv
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

True

In [5]:
# load PDF
PDF_PATH = "/Users/satvik/Desktop/Project/personal/Types-of-RAG/ Multimodal RAG/crag_paper.pdf"

loader = UnstructuredLoader(
    PDF_PATH,
    mode="elements",
    strategy="hi_res",
    extract_images_in_pdf=True,
)
elements = loader.load()

print(f"Loaded {len(elements)} elements")
for cat in sorted(set(el.metadata.get("category", "unknown") for el in elements)):
    count = sum(1 for el in elements if el.metadata.get("category") == cat)
    print(f"  {cat}: {count}")

INFO: Reading PDF for file: /Users/satvik/Desktop/Project/personal/Types-of-RAG/ Multimodal RAG/crag_paper.pdf ...


Loaded 247 elements
  FigureCaption: 6
  Footer: 1
  Formula: 1
  Header: 1
  Image: 8
  ListItem: 41
  NarrativeText: 105
  Table: 7
  Title: 32
  UncategorizedText: 45


In [6]:
elements[0].metadata.keys()

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'file_directory', 'filename', 'category', 'element_id'])

In [7]:
for element in elements:
    if element.metadata.get("category") == "Image":
        print(element.metadata.keys())
        print(element.metadata)
        break

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'image_path', 'file_directory', 'filename', 'category', 'element_id'])
{'source': '/Users/satvik/Desktop/Project/personal/Types-of-RAG/ Multimodal RAG/crag_paper.pdf', 'coordinates': {'points': ((np.float64(213.1065000888888), np.float64(1685.6428340266664)), (np.float64(213.1065000888888), np.float64(1761.8372080844445)), (np.float64(268.9823708577777), np.float64(1761.8372080844445)), (np.float64(268.9823708577777), np.float64(1685.6428340266664))), 'system': 'PixelSpace', 'layout_width': 1654, 'layout_height': 2339}, 'last_modified': '2026-06-29T15:02:35', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'image_path': '/Users/satvik/Desktop/Project/personal/Types-of-RAG/ Multimodal RAG/figures/figure-1-1.jpg', 'file_directory': '/Users/satvik/Desktop/Project/personal/Types-of-RAG/ Multimodal RAG', 'filename': 'crag_paper.pdf', 'category': 'Image', 'element_id': '9c5fdac

In [8]:
# caption images with VLM
vlm = ChatOpenAI(model="gpt-5-mini")

IMAGE_CAPTION_SYSTEM_PROMPT = """You are a document analysis assistant. Your task is to generate \
detailed, accurate descriptions of images extracted from a document. These descriptions will be \
embedded into a vector store and used for semantic retrieval, so they must capture all information \
a user might search for.

For each image, describe:
- The image type (chart, diagram, photograph, table, illustration, screenshot, etc.)
- All visible text, labels, titles, captions, and annotations
- Key data, values, trends, or patterns (especially for charts and graphs)
- The main subject and all important visual elements
- Spatial relationships and structure where relevant

Be specific and thorough. Avoid vague language."""

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def caption_image(image_path: str) -> str:
    b64 = encode_image(image_path)
    messages = [
        SystemMessage(content=IMAGE_CAPTION_SYSTEM_PROMPT),
        HumanMessage(content=[
            {"type": "text", "text": "Describe this image extracted from a document."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]),
    ]
    return vlm.invoke(messages).content

In [9]:
image_docs = []

for el in elements:
    if el.metadata.get("category") == "Image":
        image_path = el.metadata.get("image_path", "")
        if image_path:
            caption = caption_image(image_path)
            image_docs.append(Document(page_content=caption, metadata=el.metadata))

print(f"Captioned {len(image_docs)} images")

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Captioned 8 images


In [10]:
print(image_docs[-1].page_content)

Image type:
- A 2D line chart (plot) with two series, markers, gridlines, and a horizontal annotation line.

Overall layout and titles/labels:
- X-axis label: "Accuracy of retrieval"
- Y-axis label: "Accuracy of generation"
- No main chart title is present.
- Legend in the upper-right corner inside a white box showing two series:
  - a green star marker and line labeled "Self-RAG"
  - a gray diamond marker and line labeled "Self-CRAG"

Axes, ticks and ranges:
- X-axis tick labels (left to right): "69.8 (Actual)", "60", "50", "40", "30", "20", "10". These x-values are arranged left-to-right in decreasing order of retrieval accuracy (69.8 on left, 10 on right).
- Vertical dashed guide lines are drawn at each x tick.
- Y-axis tick range visible from 20 up to 70 (tick marks/scale visually spanning this range). The top of the y-axis is at 70.

Visible text and annotation:
- Blue dashed horizontal line across the plot at y ≈ 30 annotated in blue text just to the left of the line: "no retriev

In [11]:
# split text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_elements = [el for el in elements if el.metadata.get("category") != "Image"]

text_docs = splitter.split_documents(text_elements)
caption_docs = splitter.split_documents(image_docs)

all_docs = text_docs + caption_docs
print(f"Total chunks: {len(all_docs)} ({len(text_docs)} text + {len(caption_docs)} captions)")

Total chunks: 275 (249 text + 26 captions)


In [12]:
# embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [13]:
# vector store
vector_store = Chroma.from_documents(filter_complex_metadata(all_docs), embeddings)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [14]:
# retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [15]:
# RAG chain
prompt = ChatPromptTemplate.from_messages([
    ("human", "Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}"),
])

def build_messages(inputs):
    docs = inputs["docs"]
    question = inputs["question"]

    text_chunks = [d for d in docs if d.metadata.get("category") != "Image"]
    image_chunks = [d for d in docs if d.metadata.get("category") == "Image"]

    text_context = "\n\n".join(d.page_content for d in text_chunks)

    messages = prompt.format_messages(context=text_context, question=question)

    if image_chunks:
        seen_paths = set()
        image_content = []
        for doc in image_chunks:
            image_path = doc.metadata.get("image_path")
            if image_path and image_path not in seen_paths:
                seen_paths.add(image_path)
                image_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encode_image(image_path)}"},
                })

        if image_content:
            text_content = messages[-1].content
            messages[-1] = HumanMessage(content=[
                {"type": "text", "text": text_content},
                *image_content,
            ])

    return messages

llm = ChatOpenAI(model="gpt-5-mini")

chain = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(build_messages)
    | {"response": llm | StrOutputParser(), "context": RunnablePassthrough()}
)

In [16]:
# test
question = "How does Self-CRAG compares with Self-RAG as shown in the line chart. Can you explain this in a little bit more detail?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Short answer: Self-CRAG consistently beats Self-RAG at every retrieval accuracy shown, and the advantage grows as retrieval quality falls — i.e., Self-CRAG is more accurate overall and more robust to poorer retrieval.

Details from the chart
- Both curves slope downward as retrieval accuracy decreases, but the Self-CRAG curve (gray diamonds) sits above the Self-RAG curve (green stars) at every point.
- At the highest retrieval accuracy (≈69.8%) generation accuracy is about 62–63% for Self-CRAG vs ~55% for Self-RAG (≈7–8 point gap).
- At the lowest retrieval accuracy shown (≈10%) generation accuracy is about 50–53% for Self-CRAG vs ≈32–34% for Self-RAG (≈18–20 point gap).
- The dashed horizontal “no retrieval” baseline (~30%) lies near where Self-RAG falls at low retrieval accuracy, while Self-CRAG remains well above that baseline across the range.

Interpretation
- Self-CRAG not only improves absolute generation accuracy, it degrades more slowly as retrieval quality worsens. That indic

In [17]:
len(answer["context"][0].content)

2

In [18]:
question = "Computational requirements of CRAG vs Self-RAG and which was has faster execution time and can you give me the actual TFLOPS values?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


From Table 6 (generation phase only; retrieval/data processing excluded):

- CRAG: 27.2 TFLOPs per token; average execution time 0.512 s per instance.
- Self-RAG: 26.5 → 132.4 TFLOPs per token (adaptive range; lower bound ≈ RAG, upper bound much higher); average execution time 0.741 s per instance.

So CRAG has a fixed, modest computational requirement (27.2 TFLOPs/token) and is faster (0.512 s vs 0.741 s). Self-RAG can be as cheap as ~26.5 TFLOPs/token but may require up to ~132.4 TFLOPs/token in the worst cases and runs slower on average.
